# Data Preprocessing

- Raw abstract text (long)
- CLEAN the text
- remove junk characters
- CHUNK the text
- split into ~512 word pieces
- Ready for embeddings 

### Why do we chunk?
Embedding models have a token limit
You can't embed an entire paper at once
Smaller chunks = more precise retrieval
"Find the exact paragraph about tau protein"
vs
"Find the entire paper somewhere"

### Load Your Saved Papers

In [1]:
# Cell 1: Load the papers we fetched from PubMed
# We saved them in Block 1 as JSON

import json

# Load the papers
with open("../data/raw/alzheimer_papers.json", "r") as f:
    papers = json.load(f)

print(f"Loaded {len(papers)} papers")
print("\nFirst paper title:")
print(papers[0]["title"])
print("\nAbstract length (characters):")
print(len(papers[0]["abstract"]))

Loaded 10 papers

First paper title:
Serum Expression of miR-106b-3p and Its Diagnostic Significance in Alzheimer Disease.

Abstract length (characters):
133


### Clean The Text

You saved 10 papers in Block 1
Each paper has:
  - pmid      → unique ID
  - title     → paper title
  - abstract  → the long text
  - source    → "PubMed"

In Block 2 we're going to:
  - Take that abstract text
  - Clean it up
  - Split it into chunks
  - Each chunk ~512 words

Why 512?
  Because our embedding model
  "all-MiniLM-L6-v2" works best
  with text that size ✅

In [4]:
# Cell 2: Clean the abstract text
# Remove junk characters, extra spaces, encoding issues
#re.sub()     → find and replace using patterns
#\s+          → one or more whitespace characters
#\w\s\.\,     → keep letters, spaces, punctuation
#strip()      → remove spaces from start and end

import re

def clean_text(text):
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text)
    # Remove special characters but keep punctuation
    text = re.sub(r'[^\w\s\.\,\;\:\!\?\-\(\)]', ' ', text)
    # Strip leading/trailing spaces
    text = text.strip()
    return text

# Test it on first paper
sample = papers[0]["abstract"]
cleaned = clean_text(sample)

print("BEFORE cleaning:")
print(sample)
print("\nAFTER cleaning:")
print(cleaned)

BEFORE cleaning:
MicroRNAs, as key regulators in gene expression, may hold the key to understanding Alzheimer disease (AD) pathogenesis and diagnosis.

AFTER cleaning:
MicroRNAs, as key regulators in gene expression, may hold the key to understanding Alzheimer disease (AD) pathogenesis and diagnosis.


In [5]:
print("\nLength before:", len(sample))
print("Length after:", len(cleaned))
print("\nCleaning worked!" if cleaned else "Empty abstract!")


Length before: 133
Length after: 133

Cleaning worked!


### Chunking

In [6]:
# Cell 3: Split text into chunks
# Why? Embedding models work best with ~512 word pieces

def chunk_text(text, chunk_size=512, overlap=64):
    
    words = text.split()        # split into individual words
    chunks = []
    start = 0
    
    while start < len(words):
        # take chunk_size words
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        # move forward but keep overlap words
        start = end - overlap
    
    return chunks

# Test on first paper
sample_abstract = papers[0]["abstract"]
chunks = chunk_text(sample_abstract)

print(f"Abstract word count: {len(sample_abstract.split())}")
print(f"Number of chunks: {len(chunks)}")
print(f"\nChunk 1:\n{chunks[0]}")

Abstract word count: 19
Number of chunks: 1

Chunk 1:
MicroRNAs, as key regulators in gene expression, may hold the key to understanding Alzheimer disease (AD) pathogenesis and diagnosis.


### Process ALL Papers

In [7]:
# Cell 4: Chunk all 10 papers and structure the data
# This creates our final dataset ready for embeddings

all_chunks = []

for paper in papers:
    
    # Combine title + abstract for richer context
    full_text = paper["title"] + ". " + paper["abstract"]
    
    # Clean it
    cleaned = clean_text(full_text)
    
    # Chunk it
    chunks = chunk_text(cleaned)
    
    # Store each chunk with its metadata
    for i, chunk in enumerate(chunks):
        all_chunks.append({
            "pmid": paper["pmid"],
            "chunk_id": i,
            "text": chunk,
            "source": "PubMed",
            "title": paper["title"]
        })

print(f"Total papers: {len(papers)}")
print(f"Total chunks: {len(all_chunks)}")
print(f"\nSample chunk:")
print(all_chunks[0])

Total papers: 10
Total chunks: 10

Sample chunk:
{'pmid': '41773883', 'chunk_id': 0, 'text': 'Serum Expression of miR-106b-3p and Its Diagnostic Significance in Alzheimer Disease.. MicroRNAs, as key regulators in gene expression, may hold the key to understanding Alzheimer disease (AD) pathogenesis and diagnosis.', 'source': 'PubMed', 'title': 'Serum Expression of miR-106b-3p and Its Diagnostic Significance in Alzheimer Disease.'}


### Save the Chunks

In [8]:
# Cell 5: Save chunks to file
# This is the input for Block 3 (Embeddings)

import json
import os

os.makedirs("../data/processed", exist_ok=True)

with open("../data/processed/chunks.json", "w") as f:
    json.dump(all_chunks, f, indent=2)

print(f"Saved {len(all_chunks)} chunks")
print("\nSample chunk structure:")
print(json.dumps(all_chunks[0], indent=2))

Saved 10 chunks

Sample chunk structure:
{
  "pmid": "41773883",
  "chunk_id": 0,
  "text": "Serum Expression of miR-106b-3p and Its Diagnostic Significance in Alzheimer Disease.. MicroRNAs, as key regulators in gene expression, may hold the key to understanding Alzheimer disease (AD) pathogenesis and diagnosis.",
  "source": "PubMed",
  "title": "Serum Expression of miR-106b-3p and Its Diagnostic Significance in Alzheimer Disease."
}


### Check Your Saved File:

In [9]:
#data/
  ├── #raw/
  │     └── #alzheimer_papers.json    ← Block 1 output
  └── #processed/
        └── #chunks.json              ← Block 2 output 

IndentationError: unexpected indent (3102974127.py, line 2)

**BLOCK 2 COMPLETE!**

- ✅ Loaded papers from JSON
- ✅ Cleaned the text
- ✅ Chunked into 512 word pieces
- ✅ Combined title + abstract
- ✅ Saved to processed folder